In [2]:
import pandas as pd
import numpy as np

In [3]:
file_path = 'affiliate_orders_masked.csv'
df_raw = pd.read_csv(file_path)

print(f"Dataset Loaded Successfully!")
print(f"Total Raw Rows   : {df_raw.shape[0]}")
print(f"Total Raw Columns: {df_raw.shape[1]}")

Dataset Loaded Successfully!
Total Raw Rows   : 6889
Total Raw Columns: 47


In [6]:
df_raw.head(5)

,ID pesanan,ID SKU,Nama produk,ID produk,Harga,Produk terjual,Produk yang dikembalikan dananya,Nama toko,Kode toko,Mitra afiliasi,...,Bonus,Bonus mitra afiliasi,Pajak - ISR,Pajak - IVA,cedular_tax,Pajak - PIT,Dibagikan kepada mitra,Total penghasilan akhir,Tanggal pesanan,Tanggal penyelesaian pembayaran komisi
0,10981,1730706088342881879,Sepatu Tribune Away Maroon,1730706121172026967,530400.0,1,0,Toko Sepatu Bandung,IDLCPDWXGD,Affiliate_-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,265.2,31/03/2026 21:09:23,06/04/2026 17:03:45
1,14444,1730706107471988311,Sepatu Tribune Home Black,1730706116728161879,537200.0,1,0,Toko Sepatu Bandung,IDLCPDWXGD,Affiliate_-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,268.6,31/03/2026 18:54:45,05/04/2026 13:31:24
2,76840,1730706106318554711,Sepatu Tribune Home Black,1730706116728161879,514488.0,1,0,Toko Sepatu Bandung,IDLCPDWXGD,Affiliate_-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25724.4,31/03/2026 18:42:34,06/04/2026 17:39:26
3,26243,1730651001345574487,Sepatu Tribune Navy Gum,1730651021212354135,447304.0,1,0,Toko Sepatu Bandung,IDLCPDWXGD,Affiliate_-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22365.2,31/03/2026 18:31:02,07/04/2026 12:12:24
4,14352,1729677743140866647,Sepatu Retrograde Low Black White,1729677743140473431,431800.0,1,0,Toko Sepatu Bandung,IDLCPDWXGD,Affiliate_-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,215.9,31/03/2026 17:55:48,06/04/2026 13:19:05


In [7]:
df_clean = df_raw.copy()

In [8]:
df_clean['Tanggal pesanan'] = pd.to_datetime(
    df_clean['Tanggal pesanan'], format='%d/%m/%Y %H:%M:%S', errors='coerce'
)
df_clean['Tanggal penyelesaian pembayaran komisi'] = pd.to_datetime(
    df_clean['Tanggal penyelesaian pembayaran komisi'], format='%d/%m/%Y %H:%M:%S', errors='coerce'
)

In [9]:
df_clean['commission_rate_clean'] = (
    df_clean['Standar']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float) / 100.0
)
df_clean['commission_rate_clean'] = df_clean['commission_rate_clean'].fillna(0.0)

In [10]:
df_clean['Agensi'] = df_clean['Agensi'].fillna('No Agency')
df_clean['Jenis Konten'] = df_clean['Jenis Konten'].fillna('Unknown')

In [11]:
dim_products = (
    df_clean[['ID SKU', 'Nama produk', 'Harga', 'Kode toko']]
    .drop_duplicates(subset=['ID SKU'])
    .copy()
)
dim_products.columns = ['sku_id', 'product_name', 'price', 'shop_id']

In [12]:
dim_content = (
    df_clean[['ID Konten', 'Jenis Konten', 'Agensi']]
    .drop_duplicates(subset=['ID Konten'])
    .copy()
)
dim_content.columns = ['content_id', 'content_type', 'agency_id']

In [13]:
fact_orders = (
    df_clean[[
        'ID pesanan', 'Tanggal pesanan', 'ID SKU', 'ID Konten',
        'Produk terjual', 'GMV', 'Status penyelesaian pembayaran pesanan'
    ]]
    .copy()
)
fact_orders.columns = [
    'order_id', 'order_date', 'sku_id', 'content_id',
    'quantity_sold', 'gmv', 'payment_status'
]

In [14]:
dim_commission_payout = (
    df_clean[[
        'ID pesanan', 'Mitra afiliasi', 'commission_rate_clean',
        'Total penghasilan akhir', 'Status penyelesaian pembayaran pesanan',
        'Tanggal penyelesaian pembayaran komisi'
    ]]
    .copy()
)
dim_commission_payout.columns = [
    'order_id', 'affiliate_id', 'commission_rate',
    'total_earnings', 'payout_status', 'payout_date'
]

print("\n--- Star Schema Table Summary ---")
print(f"dim_products          : {dim_products.shape[0]} unique SKUs")
print(f"dim_content           : {dim_content.shape[0]} unique content IDs")
print(f"fact_orders           : {fact_orders.shape[0]} transaction rows")
print(f"dim_commission_payout : {dim_commission_payout.shape[0]} payout records")


--- Star Schema Table Summary ---
dim_products          : 531 unique SKUs
dim_content           : 284 unique content IDs
fact_orders           : 6889 transaction rows
dim_commission_payout : 6889 payout records


In [15]:
validation_df = (
    fact_orders
    .merge(dim_products, on='sku_id', how='left')
    .merge(dim_content, on='content_id', how='left')
    .merge(dim_commission_payout, on='order_id', how='left')
)

total_original_rows = len(df_raw)
total_joined_rows = len(validation_df)

print("\n--- Pipeline Validation Verdict ---")
print(f"Original Records : {total_original_rows}")
print(f"Re-joined Records: {total_joined_rows}")

if total_original_rows == total_joined_rows:
    print("STATUS: SUCCESS - 100% Data Integrity Preserved (Zero Data Loss)!")
else:
    print("STATUS: WARNING - Record Mismatch Detected!")


--- Pipeline Validation Verdict ---
Original Records : 6889
Re-joined Records: 8869
STATUS: WARNING - Record Mismatch Detected!


In [16]:
dim_products.to_csv('dim_products.csv', index=False)
dim_content.to_csv('dim_content.csv', index=False)
fact_orders.to_csv('fact_orders.csv', index=False)
dim_commission_payout.to_csv('dim_commission_payout.csv', index=False)

print("\nAll 4 relational CSV files successfully generated and ready for PostgreSQL ingestion!")


All 4 relational CSV files successfully generated and ready for PostgreSQL ingestion!
